In [ ]:
%matplotlib inline

# ---- Local package import: no pip install, no shared profile changes ----
from pathlib import Path
import sys
import glob
import os
import builtins

import numpy as np
from numpy import *  # kept for old notebook cells that use bare numpy names

import scipy.constants as sc
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pyplot import *  # kept for old notebook cells
from matplotlib.colors import LogNorm

import adios2


def find_repo_root(start=None, marker=("src", "nanoplasma_analysis")):
    """Find PIConGPU_analisis repo root by walking upward from the notebook folder."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for p in [start, *start.parents]:
        candidate = p / marker[0] / marker[1]
        if candidate.exists() and candidate.is_dir():
            return p

    raise FileNotFoundError(
        f"Could not find repo root from {start}. "
        f"Expected to find {marker[0]}/{marker[1]} in this folder or one of its parents."
    )


REPO = find_repo_root()
SRC = REPO / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nanoplasma_analysis import NanoPlasmaRun, extract_step_from_filename

print("Using repo:", REPO)
print("Using source:", SRC)

# ---- Select ONE simulation ----
path = "/p/scratch/pwfa-trojan/medina2/2026_nanoplasma/004_Neutral_Smallsteps/simOutput"
# path = "/e/scratch/jureap18/medina2/004_V1_LowDensity/simOutput"

run = NanoPlasmaRun(
    path=path,
    laser_peak_at_target=89603,
)

# Keep this variable for old cells that use `files`.
files = run.files

print("Simulation output:", path)
print("Number of bp5 files:", len(files))
print("First file:", files[0])
print("Last file:", files[-1])


In [ ]:
run.plot_particle_number(species="He_e")


In [ ]:
run.plot_charge_state_evolution(
    ion_species="He_i", Zmax=2,
    show_laser_envelope=True,
    tau_fwhm_fs=40.0, I0_Wcm2=1e15, lambda_um=0.8
)


In [ ]:
from nanoplasma_analysis import extract_step_from_filename
laser_peak_at_target=89603

base_out = "/p/scratch/pwfa-trojan/medina2/notebooks/bilder"
# extract run name from simulation directory
run_name = os.path.basename(os.path.dirname(os.path.normpath(path)))

# get dt_fs once (from first file)
with adios2.Stream(files[0], "r") as f:
    for _ in f.steps():
        t0 = extract_step_from_filename(files[0])
        dt = f.read_attribute(f"/data/{t0}/dt")
        unit_time = f.read_attribute(f"/data/{t0}/unit_time")
dt_fs = dt * unit_time * 1e15

for i in np.arange(len(files)):
    it = extract_step_from_filename(files[i])
    t_rel_fs = (it - laser_peak_at_target) * dt_fs
    png_path = os.path.join( base_out, f"{run_name}_frame_{i:05d}_{t_rel_fs:+06.0f}fs.png")    
    run.plot_field_e_i_maps(
        file_index=i,
        y_nm=300.0,
        I0_Wcm2=4e14,
        zoom_ions=True,
        ion_zoom_pad_nm=50.0,
        savepath=png_path,
        #ion_charge_density_field="He_i_all_boundElectronDensity" #defaul        ion_charge_density_field: str = "He_i_all_density",
    )
    


In [ ]:
run.plot_laser_envelope_and_electron_yield(
    species="He_e",
    tau_fwhm_fs=40.0,
    I0_Wcm2=4e14,
    lambda_um=0.8
)


In [ ]:
run.plot_kinetic_energy_spectra(
    species="He_i",
    file_indices="all",
    bins=(0, 100, 100),
    normalize=False,logy=True
)


In [ ]:
run.plot_kinetic_energy_spectra(
    species="He_e",
    file_indices=[-1],  # or "all"
    bins=(0, 100, 100),
    normalize=True,logy=False
)


In [ ]:
run.plot_tail_decay_vs_time(
    species="He_i",
    Emin_eV=0.1, Emax_eV=60,
    Ebins_max_eV=200.0, Nbins=400,
    plot_characteristic_energy=True   # set True for 1/k
)


In [ ]:
run.plot_vmi(species="He_e", file_index=-1, plane=("x","y"), Nbins=200)


In [ ]:
run.plot_angular_distribution(species="He_i", file_index=-1, axis="x")


In [ ]:
run.plot_angular_distribution_vs_time(species="He_i", axis="x", skip_before_fs=-17)


In [ ]:
run.plot_asymmetry_vs_time(species="He_i", axis="x",skip_before_fs=-17)


In [ ]:
run.plot_mean_temperature_vs_time(species="He_e")


In [ ]:
run.make_field_e_i_gif(
    out_gif="002_field_e_i_maps.gif",
    y_nm=300.0,
    I0_Wcm2=4e14,
    zoom_ions=True,
    ion_zoom_pad_nm=50.0,
    ion_charge_density_field="He_i_all_boundElectronDensity",
    fps=3,
    every=1
)
